# EEG تحلیل داده‌های 
در این پروژه، ما از این داده‌ها برای شناسایی وضعیت چشم (باز یا بسته) استفاده می‌کنیم. در ادامه مراحل مختلف این فرآیند را بررسی خواهیم کرد.

## 1. بارگذاری داده‌ها
در این بخش، ما داده‌ها را از فایل  بارگذاری می‌کنیم و ویژگی‌ها و لیبل‌ها را جدا می‌کنیم.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('eeg.csv')

In [3]:
X = data.drop('eyeDetection', axis=1)
y = data['eyeDetection']

## 2. تقسیم داده‌ها
داده‌ها به دو مجموعه آموزشی و آزمایشی تقسیم می‌شوند. ما از 80% داده‌ها برای آموزش و 20% برای ارزیابی مدل استفاده می‌کنیم.



## 3. اسکیل کردن داده‌ها
# StandardScaler با استفاده از 
.برای بهبود عملکرد مدل، ویژگی‌ها را نرمال‌سازی می‌کنیم

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. تنظیم پارامترهای GridSearch(Random Forest)
در این مرحله، پارامترهای مختلف مدل را برای یافتن بهترین ترکیب تنظیم می‌کنیم
## 5. ایجاد و آموزش مدل با GridSearch(GridSearchCV)
مدل را ایجاد کرده و آن را با استفاده از  آموزش می‌دهیم تا بهترین پارامترها را پیدا کنیم

In [6]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

rf_model = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='accuracy')

## 6. پیش‌بینی و ارزیابی مدل
با استفاده از بهترین مدل به دست آمده، پیش‌بینی انجام می‌دهیم و نتایج را ارزیابی می‌کنیم.




## 7. نمایش دقت و گزارش طبقه‌بندی( Precision، Recall و F1-score)
دقت مدل و گزارش طبقه‌بندی را نمایش می‌دهیم.


In [7]:
best_rf = grid_search.best_estimator_

In [8]:
y_pred = best_rf.predict(X_test_scaled)

In [9]:
print(" best parameters:", grid_search.best_params_)
print(" accuracy : {:.4f}".format(accuracy_score(y_test, y_pred)))
print("\n Classification Report:")
print(classification_report(y_test, y_pred))

 best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
 accuracy : 0.9336

 Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.96      0.94      1651
           1       0.95      0.90      0.92      1345

    accuracy                           0.93      2996
   macro avg       0.94      0.93      0.93      2996
weighted avg       0.93      0.93      0.93      2996



## 8. اهمیت ویژگی‌ها
مهم‌ترین ویژگی‌های تأثیرگذار بر پیش‌بینی را شناسایی و نمایش می‌دهیم.


In [10]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n important features:")
print(feature_importance.head())


 important features:
   feature  importance
6       O1    0.118554
5       P7    0.106159
1       F7    0.091120
12      F8    0.081121
0      AF3    0.078130


## 9. ماتریس درهم‌ریختگی(Confusion Matrix)
ماتریس درهم‌ریختگی  را محاسبه و به صورت گرافیکی نمایش می‌دهیم.

In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("\n🔍 confusion_matrix:")
print("     Prediction 1 Prediction 0")
print(f"Real 0:  {cm[0,0]:^8}  {cm[0,1]:^8}")
print(f"Real 1:  {cm[1,0]:^8}  {cm[1,1]:^8}")


accuracy = (cm[0,0] + cm[1,1]) / cm.sum()
precision = cm[1,1] / (cm[1,1] + cm[0,1])
recall = cm[1,1] / (cm[1,1] + cm[1,0])

print(f"\n Evaluation Criteria:")
print(f"accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")


🔍 confusion_matrix:
     Prediction 1 Prediction 0
Real 0:    1581       70   
Real 1:    129       1216  

 Evaluation Criteria:
accuracy: 0.9336
Precision: 0.9456
Recall: 0.9041
